# Reproducing Girardeau, Inema & Buzsaki (2017) from dandiset 000061

Paper. Girardeau G, Inema I, Buzsaki G (2017). Reactivations of emotional memory in the
hippocampus-amygdala system during sleep. *Nature Neuroscience* 20, 1634-1642.
[doi:10.1038/nn.4637](https://doi.org/10.1038/nn.4637). The article is not open access (it is not
in PubMed Central, and its HAL record holds metadata only), so this attempt used the abstract
alone (PubMed 28892057). Every claim is a sentence of the abstract, none has a reported number,
and every method choice was fixed in the claims file before the analysis.

Dandiset. [DANDI:000061](https://dandiarchive.org/dandiset/000061/0.250624.0426), version
0.250624.0426. The abstract cites no DANDI version. There are two published versions,
0.210812.1517 and 0.250624.0426; they hold the same 40 files (same blobs and sizes) and differ only
in metadata (contributor affiliations and ORCIDs, and a `relatedResource` link to the paper, added
in 2025), none of which the analysis uses. The most recent published version is pinned.

What the dandiset holds. 40 NWB files, one per recording day, from four rats (Rat08 to
Rat11): sorted units with a region label, sleep states, epochs, and event times of hippocampal
ripples, air puffs and rewards; some files also hold raw voltage, LFP and video. There is no
position tracking; the files hold the raw video frames, from which position is not derived.

Code. The dandiset links no analysis code, and the abstract names none. The parameters are
common choices, listed in the helper module `hpcbla.py` and the claims file.

| Claim | Tested here |
|---|---|
| `hpc-bla-reactivation-post-sleep`: joint hippocampus and BLA patterns of the run reactivate in post-sleep Non-REM | tested |
| `reactivation-peaks-at-ripples`: the reactivations peak during sharp wave-ripples | tested |
| `bla-ripple-up-subgroup`: a subgroup of BLA cells is positively modulated during ripples | tested |
| `reactivation-involves-ripple-up-cells`: the reactivations involve those BLA cells | tested |
| `aversive-direction-stronger`: reactivation is stronger for the air-puff direction than the safe one | tested |
| `rats-learn-puff-location`: rats learned the location of the air puff | blocked (too large: position could only come from tracking the raw video frames, see the data check) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import hpcbla

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

notebook_started = started = time.time()
sessions = sorted(hpcbla.load_all(), key=lambda s: s["path"])
print(f"loaded {len(sessions)} files from dandiset {hpcbla.DANDISET_ID} version "
      f"{hpcbla.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
from dandi.dandiapi import DandiAPIClient

with DandiAPIClient() as client:
    meta = client.get_dandiset(hpcbla.DANDISET_ID, hpcbla.DANDISET_VERSION).get_raw_metadata()
summary = meta["assetsSummary"]
print("Archive summary: files", summary["numberOfFiles"], "; subjects", summary["numberOfSubjects"],
      "; species", [s["name"] for s in summary["species"]], "; measured", summary["variableMeasured"])

rows = []
for s in sessions:
    u = s["units"]
    pre, post, run = hpcbla.sleep_epochs(s)
    region = pd.Series(u["region"]).value_counts() if u else pd.Series(dtype=int)
    rows.append({
        "session": s["session"], "rat": s["rat"], "units": 0 if u is None else len(u["region"]),
        "hpc": int(sum(region.get(k, 0) for k in hpcbla.HPC_LABELS)),
        "bla": int(sum(region.get(k, 0) for k in hpcbla.BLA_LABELS)),
        "bla_complex": int(sum(region.get(k, 0) for k in hpcbla.BLA_COMPLEX_LABELS)),
        "states": s["states"] is not None,
        "ripples": 0 if s["ripples"] is None else len(s["ripples"]["peak"]),
        "puffs": len(s["events"]["puf"][1]) if "puf" in s["events"] else 0,
        "rewards": len(s["events"]["lrw"][1]) + len(s["events"]["rrw"][1]),
        "pre_sleep_s": round(float(np.diff(pre).sum())), "post_sleep_s": round(float(np.diff(post).sum())),
        "run_s": None if run is None else round(run[1] - run[0]),
        "lfp": s["lfp"] is not None, "raw": s["raw"] is not None, "videos": len(s["videos"]),
        "trials": None if s["trials"] is None else len(s["trials"][0])})
info = pd.DataFrame(rows)
print("\nSubjects:", sorted(info.rat.unique()), "; files per rat:", info.rat.value_counts().sort_index().to_dict())
print("Species, age, sex:", sorted({(s["species"], s["age"], s["sex"]) for s in sessions}))
print("\n", info.to_string(index=False))
print("\nFiles without a units table:", info.session[info.units == 0].tolist())
print("Files without sleep states:", info.session[~info.states].tolist())
print("Files without ripple events:", info.session[info.ripples == 0].tolist())
print("Rats with hippocampal units:", sorted(info.rat[info.hpc > 0].unique()))
print("Files with raw voltage:", int(info.raw.sum()), "; with LFP:", int(info.lfp.sum()), "; with video:", int((info.videos > 0).sum()))

The archive's summary says four subjects, which matches the four rats in the files, all male Long
Evans rats given as 3 months old. Seven files have no units table, sixteen have no sleep states, and
the Rat09 files have no ripple events and no hippocampal units, so Rat09 cannot enter any analysis
that needs both regions. Raw voltage is in 16 of the 40 files; the LFP and video are in all of them.

In [ ]:
labels = pd.Series(np.concatenate([s["units"]["label"] for s in sessions if s["units"]])).value_counts()
print("Unit location labels over all files (as stored):")
print(labels.to_string())
print("\nElectrode table locations:", sorted({x for s in sessions for x in s["electrode_locations"]}))
print("Files whose units table has no location column:",
      [s["session"] for s in sessions if s["units"] and not s["units"]["has_location_column"]])
print("\nUsed as hippocampus:", sorted(hpcbla.HPC_LABELS), "; as BLA:", sorted(hpcbla.BLA_LABELS),
      "; wider BLA variant:", sorted(hpcbla.BLA_COMPLEX_LABELS), "(case ignored)")

The unit labels are short codes with no description in the files or the dandiset metadata. Some
appear in two cases (`bla` and `BLA`, `hpc` and `HIP`, `pir` and `PIR`); in Rat08-20130717 the
upper-case labels are on shanks 13 to 20 and the lower-case ones on shanks 1 to 12, so the case
marks a probe, not a duplicate (no two units of that file share a spike count and first spike;
see the duplicate check below). Others (`ladl`, `blv`, `bmp`, `den`, `ven`, `cen`, `pir`,
`plc01`, `PLCo1`, `cpu`, `no label`) are not defined anywhere. The electrode table gives every
channel the location `unknown`. Rat09-20140328 has no location column at all.

In [ ]:
rows = []
for s in sessions:
    end_raw = max(b for _, _, b in s["epochs_raw"])
    lfp_s = s["lfp"][0][0] / s["lfp"][1] if s["lfp"] else np.nan
    rows.append({"session": s["session"], "epochs_last_stop_raw": end_raw,
                 "lfp_duration_s": round(lfp_s, 1), "ratio": round(end_raw / lfp_s, 3),
                 "last_spike_s": round(s["units"]["last_spike"], 1) if s["units"] else np.nan,
                 "lfp_rate_hz": s["lfp"][1] if s["lfp"] else np.nan,
                 "states_span_s": (min(a for _, a, _ in s["states"]), max(b for _, _, b in s["states"]))
                 if s["states"] else None})
times = pd.DataFrame(rows)
print("Epoch table column description:", sorted({s["epoch_unit_stated"] for s in sessions}))
print(times.to_string(index=False))
print("\nRatio of the epoch table's last stop to the LFP duration: min", times.ratio.min(), "max", times.ratio.max())
print("Epoch tags:", sorted({t for s in sessions for t, _, _ in s["epochs"]}))
print("Files that tag sleep only as 'sleep':", {s["session"]: [t for t, _, _ in s["epochs"]] for s in sessions
                                               if "presleep" not in [t for t, _, _ in s["epochs"]] and "postsleep" not in [t for t, _, _ in s["epochs"]]})

The epoch table says its times are in seconds, but its last stop time is 1000 times the LFP
duration (and the last spike time) in every file: the epochs are stored in milliseconds. The
helper divides them by 1000. Sleep states, spikes, ripples, puffs and rewards are in seconds and
span the session. Three files tag their sleep epochs only as `sleep`; there the sleep before the
first run is taken as pre-sleep and the sleep after it as post-sleep. Of these only
Rat08-20130708 enters the analysis (the other two have no units).

In [ ]:
rows = []
for s in sessions:
    eps = {t: (a, b) for t, a, b in s["epochs"]}
    for name, (n_frames, first, last, unit) in s["videos"].items():
        tag = name.rsplit("-", 1)[-1]
        a, b = eps.get(tag, (np.nan, np.nan))
        rows.append({"session": s["session"], "video": tag, "frames": n_frames, "stated_unit": unit,
                     "first_ts": round(first, 1), "last_ts": round(last, 1),
                     "video_span_ts/1000": round((last - first) / 1000, 1), "epoch_start_s": round(a, 1),
                     "epoch_duration_s": round(b - a, 1)})
videos = pd.DataFrame(rows)
print(videos.head(12).to_string(index=False))
print("...", len(videos), "videos in all")
ok = np.isfinite(videos.epoch_start_s)
print("Videos whose first timestamp / 1000 is within 5 s of their epoch's start:",
      int((np.abs(videos.first_ts[ok] / 1000 - videos.epoch_start_s[ok]) < 5).sum()), "of", int(ok.sum()))
print("Videos whose span / 1000 is within 5% of their epoch's duration:",
      int((np.abs(videos["video_span_ts/1000"][ok] / videos.epoch_duration_s[ok] - 1) < 0.05).sum()), "of", int(ok.sum()))
rows = []
for s in sessions:
    if s["trials"] is None:
        continue
    a, b, desc = s["trials"]
    _, _, run = hpcbla.sleep_epochs(s)
    rows.append({"session": s["session"], "trials": len(a), "description": desc,
                 "median_duration_s": round(float(np.median(b - a)), 1),
                 "fraction_in_run": round(float(np.mean((a >= run[0]) & (a < run[1]))), 2)})
print("\nTrials tables:")
print(pd.DataFrame(rows).to_string(index=False))

The video timestamps are stated in seconds but are in milliseconds, and they are not on the
session clock: the first timestamp matches the start of the epoch a video is named after in 1 of
114 videos, while the spans match the epochs' durations in 93 of 114 (the run videos of the early
Rat08 files span about half of their run epoch). The videos are not used here. The `trials` table of the Rat08 and Rat09 files is described only as "experimental
trials"; its intervals last about 3 to 5 s and fall partly outside the run, so they are not laps.
It is not used.

In [ ]:
# Video storage: frame shape and compression of every video series, the size of the run videos,
# and the time taken to stream frames of one run video.
storage = pd.DataFrame([{"session": s["session"], "video": k.rsplit("-", 1)[-1], "frames": shape[0],
                         "frame_shape": shape[1:], "dtype": dt, "compression": comp,
                         "uncompressed_gb": np.prod(shape) / 1e9}
                        for s in sessions for k, (shape, dt, comp) in s["video_storage"].items()])
print("Video series:", len(storage), "; frame shapes:", sorted(set(storage.frame_shape)), "; dtypes:", sorted(set(storage.dtype)),
      "; compression:", sorted(set(map(str, storage.compression))))
run_videos = storage[storage.video == "run"]
print("Run videos:", len(run_videos), "in", run_videos.session.nunique(), "files;", int(run_videos.frames.sum()), "frames,",
      round(float(run_videos.uncompressed_gb.sum())), "GB uncompressed")
first = sessions[0]
name = next(k for k in first["video_storage"] if k.endswith("-run"))
seconds, frame_bytes = hpcbla.time_frame_reads(dict(hpcbla.asset_urls())[first["path"]], name, 20)
print(f"Streaming 20 frames of {name}: {seconds:.1f} s ({seconds / 20:.2f} s per frame of {frame_bytes} bytes)")
print(f"At that rate the run frames alone would take about {run_videos.frames.sum() * seconds / 20 / 3600:.0f} hours to stream")


The videos are stored as raw frames (360 by 640 pixels, three colours) inside the NWB files,
compressed with gzip, with no position derived from them. Tracking the rat in the run videos would
mean streaming the frames printed above and running a tracker, which does not fit the time limit;
their timestamps would also have to be aligned to the session clock first (see above). This is
why the behaviour claim is blocked as too large.

In [ ]:
rows = []
for s in sessions:
    u = s["units"]
    if u and u["unsorted"]:
        rows.append({"session": s["session"], "unsorted_units": u["unsorted"],
                     "falls_back_after_spike": [x for i in u["unsorted"] for x in u["unsorted_at"][i]],
                     "unit0_last_spike_s": round(u["unit0_last"], 2), "unit1_first_spike_s": round(u["unit1_first"], 2),
                     "units_left_out": u["mixed"]})
print(pd.DataFrame(rows).to_string(index=False))
print(hpcbla.MIXED_UNITS_NOTE)

In ten files the spike times of unit 1 are not sorted: they fall back once, always after 65536,
131072, 196608, 327680 or 458752 spikes (multiples of 65536), and unit 0 stops part way through the
session at the time where unit 1 starts. The first part of unit 1 is the end of unit 0, so the
index that separates them is wrong. Units 0 and 1 of these files are left out.

In [ ]:
# Clock checks. (1) Ripples against the sleep states, with the states shifted by 600 s as the
# contrast. (2) Hippocampal multi-unit rate within 25 ms of ripple peaks against 1 s away.
rows = []
for s in sessions:
    if s["states"] is None or s["ripples"] is None or s["units"] is None:
        continue
    peaks = s["ripples"]["peak"]
    r = {"session": s["session"], "ripples_ordered": s["ripples"]["ordered"],
         "channels": ",".join(s["ripples"]["channels"]),
         "ripple_ms_median": round(1000 * float(np.median(s["ripples"]["stop"] - s["ripples"]["start"])), 1)}
    for label in ("Non-REM", "REM", "Awake"):
        iv = hpcbla.intervals_of(s, label)
        dur = np.diff(iv).sum()
        r[f"{label}_hz"] = round(np.count_nonzero(hpcbla.in_intervals(peaks, iv)) / dur, 3)
        r[f"{label}_hz_shift"] = round(np.count_nonzero(hpcbla.in_intervals(peaks, iv + 600)) / dur, 3)
    u = s["units"]
    hpc = [i for i in u["trains"] if u["region"][i] in hpcbla.HPC_LABELS]
    if hpc:
        spikes = np.sort(np.concatenate([u["trains"][i] for i in hpc]))
        def rate(t, w=0.025):
            return (np.searchsorted(spikes, t + w) - np.searchsorted(spikes, t - w)).mean() / (2 * w)
        r["hpc_hz_at_peak"] = round(rate(peaks), 1)
        r["hpc_hz_peak+1s"] = round(rate(peaks + 1), 1)
    rows.append(r)
clock = pd.DataFrame(rows)
print(clock.to_string(index=False))
print("\nRipples per s of Non-REM: min", clock["Non-REM_hz"].min(), "max", clock["Non-REM_hz"].max(), "; REM max", clock["REM_hz"].max())
print("Median over files: ripples per s in Non-REM", clock["Non-REM_hz"].median(),
      "(states shifted:", clock["Non-REM_hz_shift"].median(), "); in REM", clock["REM_hz"].median(),
      "(shifted:", clock["REM_hz_shift"].median(), ")")
print("Median ratio of hippocampal rate at ripple peaks to 1 s later:",
      round(float((clock.hpc_hz_at_peak / clock["hpc_hz_peak+1s"]).median()), 2), "; files with hippocampal units:",
      int(clock.hpc_hz_at_peak.notna().sum()))

In every file with states and ripples, ripples occur at 0.33 to 1.0 per second of Non-REM and at
most 0.19 per second of REM (median 0.022); with the states shifted by 600 s the two rates move
toward each other (0.37 and 0.32). Hippocampal
units fire several times faster within 25 ms of a ripple peak than 1 s later. The ripple
annotations come in ordered start, peak and stop triplets on one channel per file. So the states,
the ripples and the spikes share one clock, as far as these contrasts show.

In [ ]:
# (3) Lap direction decoded from hippocampal rates in each lap (leave one lap out, nearest
# centroid), with the true laps and with laps shifted by half their own duration.
def decode_direction(s, half_shift=False):
    u = s["units"]
    ids = [i for i in u["trains"] if u["region"][i] in hpcbla.HPC_LABELS]
    laps = hpcbla.run_laps(s)
    if len(ids) < 5 or len(laps) < 6:
        return None
    if half_shift:
        laps = [(a + (b - a) / 2, b + (b - a) / 2, d) for a, b, d in laps]
    x = np.array([[np.count_nonzero((u["trains"][i] >= a) & (u["trains"][i] < b)) / (b - a) for i in ids]
                  for a, b, _ in laps])
    y = np.array([d == "LR" for _, _, d in laps])
    x = np.log1p(x)
    x = (x - x.mean(0)) / (x.std(0) + 1e-9)
    hits = 0
    for k in range(len(y)):
        keep = np.arange(len(y)) != k
        c1, c0 = x[keep & y].mean(0), x[keep & ~y].mean(0)
        hits += (np.sum((x[k] - c1) ** 2) < np.sum((x[k] - c0) ** 2)) == y[k]
    return hits / len(y), len(y)

rows = []
for s in sessions:
    if s["units"] is None:
        continue
    true, shifted = decode_direction(s), decode_direction(s, True)
    if true:
        rows.append({"session": s["session"], "laps": true[1], "accuracy": round(true[0], 3),
                     "accuracy_half_lap_shift": round(shifted[0], 3)})
dec = pd.DataFrame(rows)
print(dec.to_string(index=False))
print("Median accuracy", dec.accuracy.median(), "; with laps shifted by half a lap", dec.accuracy_half_lap_shift.median(),
      "; chance 0.5")

Hippocampal rates tell the lap direction apart in nearly every lap with the laps built from the
reward times, and less well when the laps are shifted by half their length. The reward events and
the spikes share one clock, as far as this contrast shows; it does not test the timing to better
than a fraction of a lap.

In [ ]:
# (4) Air puffs: pooled rate in the 0.5 s after each puff against the 0.5 s before, in BLA
# (wider labels) and hippocampus, with the puff times shifted by 30 s as the contrast.
rows = []
for s in sessions:
    u = s["units"]
    if not u or "puf" not in s["events"] or len(s["events"]["puf"][1]) < 5:
        continue
    puffs = s["events"]["puf"][1]
    direction, counts = hpcbla.puff_direction(s)
    row = {"session": s["session"], "puffs": len(puffs), "direction": direction, **counts}
    for name, labels_ in (("bla", hpcbla.BLA_COMPLEX_LABELS), ("hpc", hpcbla.HPC_LABELS)):
        ids = [i for i in u["trains"] if u["region"][i] in labels_]
        if len(ids) < 3:
            continue
        spikes = np.sort(np.concatenate([u["trains"][i] for i in ids]))
        def change(t):
            after = (np.searchsorted(spikes, t + 0.5) - np.searchsorted(spikes, t)).sum()
            before = (np.searchsorted(spikes, t) - np.searchsorted(spikes, t - 0.5)).sum()
            return (after - before) / max(before, 1)
        row[f"{name}_change"] = round(change(puffs), 3)
        row[f"{name}_change_shift30"] = round(change(puffs + 30), 3)
    rows.append(row)
puff = pd.DataFrame(rows)
print(puff.to_string(index=False))
print("\nMedian relative change after puffs, BLA:", puff.bla_change.median(), "(shifted:", puff.bla_change_shift30.median(),
      "); hippocampus:", puff.hpc_change.median(), "(shifted:", puff.hpc_change_shift30.median(), ")")
print("Files with BLA change above 0.3:", int((puff.bla_change > 0.3).sum()), "; with shifted times:",
      int((puff.bla_change_shift30 > 0.3).sum()))
print("Puffs inside laps of the session's air-puff direction, in these files:", int(sum(puff[d].where(puff.direction == d, 0).sum() for d in ("LR", "RL"))),
      "of", int(puff.puffs.sum()))

In the 31 files above (units and at least 5 puffs), nearly all puffs (927 of 957) fall inside
laps of one direction in each session, which gives the air-puff direction; in the 15 sessions
used for the direction claim, 295 of 304 do (see that section). The puff side changes between days in the same rat. The pooled response to
puffs is small in most files (median change 0.04 in BLA and in hippocampus); four files show a
BLA increase above 0.3, against one with the shifted times. This is
consistent with the puffs sharing the spike clock but does not show it for every file.

In [ ]:
# Recordings deposited twice: compare unit spike counts and first spikes, ripple times and puff
# times between every pair of files, and units within a file.
def signature(s):
    u = s["units"]
    return {"units": None if u is None else tuple(zip(u["n_spikes"].tolist(), np.round(u["first_spike"], 4).tolist())),
            "ripples": None if s["ripples"] is None else tuple(np.round(s["ripples"]["peak"][:50], 4)),
            "puffs": tuple(np.round(s["events"]["puf"][1], 4)) if "puf" in s["events"] else None,
            "rewards": tuple(np.round(np.sort(np.concatenate([s["events"]["lrw"][1], s["events"]["rrw"][1]])), 4))}
sigs = {s["session"]: signature(s) for s in sessions}
names = list(sigs)
same = [(a, b, k) for i, a in enumerate(names) for b in names[i + 1:] for k in sigs[a]
        if sigs[a][k] is not None and sigs[a][k] == sigs[b][k]]
print("Pairs of files sharing units, ripples, puffs or rewards:", same if same else "none")
within = {s["session"]: len(sigs[s["session"]]["units"]) - len(set(sigs[s["session"]]["units"]))
          for s in sessions if s["units"]}
print("Units sharing spike count and first spike within a file:", {k: v for k, v in within.items() if v} or "none")

No two files share their units, ripples, puffs or rewards, and no two units of a file share a
spike count and first spike, so no recording was deposited twice. Units are not matched across
days; each session is analysed on its own and results are given per session and per rat.

In [ ]:
rows = []
for s in sessions:
    if s["units"] is None:
        continue
    hpc, bla = hpcbla.unit_sets(s)
    hpc_w, bla_w = hpcbla.unit_sets(s, hpcbla.BLA_COMPLEX_LABELS)
    rows.append({"session": s["session"], "rat": s["rat"], "hpc_used": len(hpc), "bla_used": len(bla),
                 "bla_complex_used": len(bla_w), "usable": hpcbla.usable(s),
                 "usable_wider_bla": hpcbla.usable(s, hpcbla.BLA_COMPLEX_LABELS),
                 "puff_direction": hpcbla.puff_direction(s)[0]})
use = pd.DataFrame(rows)
print(use.to_string(index=False))
used = [s for s in sessions if hpcbla.usable(s)]
print("\nSessions used:", len(used), {r: sum(s["rat"] == r for s in used) for r in sorted({s["rat"] for s in used})})
print("Sessions with the wider BLA labels:", int(use.usable_wider_bla.sum()))
print("Units used (hippocampus, BLA):", int(use.hpc_used[use.usable].sum()), int(use.bla_used[use.usable].sum()))

Under the claims file's rule (sleep states, ripples, a sleep epoch before and after the run, and
at least five hippocampal and five BLA units at 0.1 Hz or more in the run), 15 sessions from three
rats are used: nine of Rat08, three of Rat10 and three of Rat11, with 341 hippocampal and 483 BLA
units. With the wider BLA labels, 21 sessions qualify.

In [ ]:
# Epoch boundaries (s) of the sessions used: every run epoch, the one used, and the sleep epochs.
rows = []
for s in used:
    pre, post, run = hpcbla.sleep_epochs(s)
    rows.append({"session": s["session"], "run_epochs": [(round(a), round(b)) for t, a, b in s["epochs"] if t == "run"],
                 "run_used": (round(run[0]), round(run[1])), "pre_sleep": [(round(a), round(b)) for a, b in pre],
                 "post_sleep": [(round(a), round(b)) for a, b in post]})
print(pd.DataFrame(rows).to_string(index=False))


Each session used has one run epoch, except Rat08-20130708, which has a second run of about
600 s after the post-sleep; there the first run is used, and its post-sleep is the sleep between
the two runs. Pre- and post-sleep are the epochs next to the run used.

## hpc-bla-reactivation-post-sleep

> Coordinated hippocampus and BLA activity patterns of the training run are reactivated during
> non-REM sleep after training, more than during non-REM sleep before it.

Joint patterns are the singular vectors of the hippocampus by BLA cross-correlation matrix of the
run (50 ms bins); a pattern is significant when its singular value exceeds the 95th percentile of
the largest one over 200 shuffles of the BLA trains. Reactivation strength is
(z_hpc . u)(z_bla . v) in each Non-REM bin.

In [ ]:
rng = np.random.default_rng(61001)
started = time.time()
main = {s["session"]: hpcbla.reactivation(s, rng) for s in used}
patterns = pd.DataFrame([r for out in main.values() for r in out["rows"]])
patterns["change"] = patterns.post - patterns.pre
sig = patterns[patterns.significant].copy()
print(f"computed in {time.time() - started:.0f} s")
print("Significant patterns:", len(sig), "of", len(patterns), "; sessions with at least one:",
      sig.session.nunique(), "of", len(used))
per_session = sig.groupby(["rat", "session"]).agg(patterns=("change", "size"), pre=("pre", "median"),
                                                  post=("post", "median"), change=("change", "median"),
                                                  positive=("change", lambda x: int((x > 0).sum())))
print(per_session.round(4).to_string())

def wilcoxon(x):
    x = np.asarray(x)
    return stats.wilcoxon(x).pvalue if len(x) > 0 and np.any(x != 0) else np.nan

print("\nPooled: median post minus pre", round(sig.change.median(), 4), "; positive", int((sig.change > 0).sum()),
      "of", len(sig), "; Wilcoxon p =", f"{wilcoxon(sig.change):.2g}")
by_rat = sig.groupby("rat").change.agg(["size", "median", lambda x: int((x > 0).sum()), wilcoxon])
by_rat.columns = ["patterns", "median_change", "positive", "wilcoxon_p"]
print(by_rat.to_string())
print("\nReference: non-significant patterns, median post minus pre",
      round(patterns[~patterns.significant].change.median(), 4), "over", int((~patterns.significant).sum()))
print("Session medians above zero:", int((per_session.change > 0).sum()), "of", len(per_session),
      "; Wilcoxon over session medians p =", f"{wilcoxon(per_session.change):.2g}")

In [ ]:
# Variants: 25 and 100 ms bins, the wider BLA labels, and the first pattern of every session.
rng_variants = np.random.default_rng(61002)
rows = []
for name, kwargs, sess in (("25 ms bins", {"bin_s": 0.025}, used), ("100 ms bins", {"bin_s": 0.1}, used),
                           ("wider BLA labels", {"bla_labels": hpcbla.BLA_COMPLEX_LABELS},
                            [s for s in sessions if s["units"] and hpcbla.usable(s, hpcbla.BLA_COMPLEX_LABELS)])):
    df = pd.DataFrame([r for s in sess for r in hpcbla.reactivation(s, rng_variants, keep_traces=False, **kwargs)["rows"]])
    df["change"] = df.post - df.pre
    for subset, d in (("significant", df[df.significant]), ("first pattern", df[df.pattern == 0])):
        rows.append({"variant": name, "patterns": subset, "n": len(d), "sessions": d.session.nunique(),
                     "median_change": round(d.change.median(), 4), "positive": int((d.change > 0).sum()),
                     "p": wilcoxon(d.change),
                     "rats_median_above_0": int((d.groupby("rat").change.median() > 0).sum()), "rats": d.rat.nunique()})
first = patterns[patterns.pattern == 0]
rows.append({"variant": "main (50 ms)", "patterns": "first pattern", "n": len(first), "sessions": first.session.nunique(),
             "median_change": round(first.change.median(), 4), "positive": int((first.change > 0).sum()),
             "p": wilcoxon(first.change), "rats_median_above_0": int((first.groupby("rat").change.median() > 0).sum()),
             "rats": first.rat.nunique()})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
for rat, d in sig.groupby("rat"):
    ax.scatter(d.pre, d.post, s=14, label=rat)
lim = [min(sig.pre.min(), sig.post.min()), max(sig.pre.max(), sig.post.max())]
ax.plot(lim, lim, "k--", lw=0.8)
ax.set_xlabel("pre-sleep Non-REM strength")
ax.set_ylabel("post-sleep Non-REM strength")
ax.legend(fontsize=8)
plt.show()
print("Plotted ranges: pre", round(sig.pre.min(), 4), "to", round(sig.pre.max(), 4), "; post",
      round(sig.post.min(), 4), "to", round(sig.post.max(), 4), "; no point outside the axes")

**Result: reproduced.** All 15 sessions have at least one significant joint pattern (36 in all).
Over these patterns the post-sleep minus pre-sleep Non-REM strength has a median of 0.0119, is
positive for 28 of 36, and the Wilcoxon p is 3.6e-05. The median is above zero in all three rats
(Rat08 0.0170, 18 of 21; Rat10 0.0024, 5 of 8; Rat11 0.0190, 5 of 7), though in Rat10 and Rat11 alone
the test is not significant (p = 0.74 and 0.11). Non-significant patterns change by a median of
0.0001. Every variant (25 and 100 ms bins, the wider BLA labels, the first pattern of each
session) gives a positive median with p of 0.0084 or less.

## reactivation-peaks-at-ripples

> The hippocampus and BLA reactivations peak during hippocampal sharp wave-ripples.

The ripple-peak-triggered mean of each significant pattern's strength in post-sleep Non-REM, from
-1 to +1 s in 50 ms bins, and reading (b): the post minus pre change in bins that overlap a ripple
against bins outside ripples.

In [ ]:
traces = pd.DataFrame([t for out in main.values() for t in out["traces"]])
lags_s = (np.arange(-20, 21)) * hpcbla.BIN_S
curves = np.vstack(traces.curve)
curves_pre = np.vstack(traces.curve_pre)
mean_curve = curves.mean(0)
print("Patterns:", len(traces), "; post-sleep Non-REM ripples per session: median", int(traces.groupby("session").n_ripples.first().median()))
print("Reading (a): the mean curve is highest at lag", round(lags_s[np.argmax(mean_curve)], 3), "s")
print(pd.DataFrame({"lag_s": lags_s, "post": mean_curve, "pre": curves_pre.mean(0)}).iloc[14:27].round(4).to_string(index=False))
flank = (np.abs(lags_s) >= 0.5 - 1e-9)
peak_minus_flank = curves[:, 20] - curves[:, flank].mean(1)
print("Peak bin minus flanks (0.5 to 1 s): median", round(float(np.median(peak_minus_flank)), 4), "; positive",
      int((peak_minus_flank > 0).sum()), "of", len(peak_minus_flank), "; Wilcoxon p =", f"{wilcoxon(peak_minus_flank):.2g}")
print("Ratio of the mean curve at the peak bin to its flanks:", round(float(mean_curve[20] / mean_curve[flank].mean()), 2))
traces["peak_minus_flank"] = peak_minus_flank
traces["argmax_lag_s"] = lags_s[np.argmax(curves, 1)]
print("Patterns whose own curve peaks within 100 ms of the ripple:", int((np.abs(traces.argmax_lag_s) <= 0.1 + 1e-9).sum()), "of", len(traces))
print(traces.groupby("rat").agg(patterns=("pattern", "size"), median_peak_minus_flank=("peak_minus_flank", "median"),
                                positive=("peak_minus_flank", lambda x: int((x > 0).sum()))).round(4).to_string())
pre_pmf = curves_pre[:, 20] - curves_pre[:, flank].mean(1)
print("Reference, pre-sleep: peak bin minus flanks median", round(float(np.median(pre_pmf)), 4),
      "; post minus pre of this difference: median", round(float(np.median(peak_minus_flank - pre_pmf)), 4),
      ", positive", int((peak_minus_flank - pre_pmf > 0).sum()), "of", len(traces), ", Wilcoxon p =",
      f"{wilcoxon(peak_minus_flank - pre_pmf):.2g}")

In [ ]:
traces["change_rip"] = traces.post_rip - traces.pre_rip
traces["change_out"] = traces.post_out - traces.pre_out
d = traces.change_rip - traces.change_out
print("Reading (b): post minus pre in ripple bins, median", round(traces.change_rip.median(), 4),
      "; outside ripples", round(traces.change_out.median(), 4))
print("Difference (ripple minus outside): median", round(float(d.median()), 4), "; positive", int((d > 0).sum()), "of", len(d),
      "; Wilcoxon p =", f"{wilcoxon(d):.2g}")
print(traces.assign(d=d).groupby("rat").d.agg(["size", "median", lambda x: int((x > 0).sum())]).round(4).to_string())
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(lags_s, mean_curve, label="post-sleep")
ax.plot(lags_s, curves_pre.mean(0), label="pre-sleep")
ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("time from ripple peak (s)")
ax.set_ylabel("mean reactivation strength")
ax.legend()
plt.show()
print("Plotted range:", round(float(min(mean_curve.min(), curves_pre.mean(0).min())), 4), "to",
      round(float(max(mean_curve.max(), curves_pre.mean(0).max())), 4))

In [ ]:
# Session level: the per-pattern measures above averaged within each session, tested over sessions.
traces["d_rip"] = traces.change_rip - traces.change_out
traces["pmf_post_minus_pre"] = traces.peak_minus_flank - pre_pmf
by_session = traces.groupby(["rat", "session"])[["peak_minus_flank", "pmf_post_minus_pre", "d_rip"]].mean()
print(by_session.round(4).to_string())
for col, label in (("peak_minus_flank", "reading (a), peak minus flanks"),
                   ("pmf_post_minus_pre", "reading (a), post minus pre of peak minus flanks"),
                   ("d_rip", "reading (b), ripple minus outside change")):
    x = by_session[col]
    print(f"{label}: median over {len(x)} sessions {x.median():.4f}, positive {int((x > 0).sum())}, Wilcoxon p = {wilcoxon(x):.2g}")


**Result: reproduced.** Under reading (a), the mean ripple-triggered strength in post-sleep is
highest in the bin of the ripple peak, 4.36 times its mean 0.5 to 1 s away; the peak bin exceeds
the flanks in 30 of 36 patterns (median 0.103, p = 6e-06), with a positive median in every rat,
and in 13 of 15 sessions at session level (p = 0.00031). Much of this peak is already present in
pre-sleep (0.136 at the ripple, against 0.223 in post-sleep), so reading (a) alone does not
separate reactivation of the run from co-activation at ripples in general. The part that does,
the peak over flanks being larger in post-sleep than in pre-sleep, holds in 23 of 36 patterns
(p = 0.022) and 10 of 15 sessions (p = 0.03). Under reading (b), the post minus pre change is
larger in ripple bins than outside them (median difference 0.0537, 25 of 36 patterns, p =
0.0033; 10 of 15 sessions, p = 0.03).

## bla-ripple-up-subgroup

> A subgroup of BLA cells is positively modulated during hippocampal sharp wave-ripples.

Spikes of each BLA unit in the windows of the Non-REM ripples of pre- and post-sleep, against 500
shifts of all windows together by 0.5 to 5 s. The chance level is the fraction selected with the
real windows replaced by the windows shifted by 7 s.

In [ ]:
rng_rip = np.random.default_rng(61003)
started = time.time()
rows = []
for s in sessions:
    u = s["units"]
    if s["states"] is None or s["ripples"] is None or u is None:
        continue
    pre, post, _ = hpcbla.sleep_epochs(s)
    nrem = hpcbla.intersect(hpcbla.intervals_of(s, "Non-REM"), np.vstack([pre, post]))
    r = s["ripples"]
    keep = hpcbla.in_intervals(r["peak"], nrem)
    start, stop = r["start"][keep], r["stop"][keep]
    end = max(b for _, _, b in s["epochs"])
    used_bla = set(hpcbla.unit_sets(s)[1])
    for i in u["trains"]:
        if u["region"][i] not in hpcbla.BLA_LABELS:
            continue
        real, chance = hpcbla.ripple_modulation(u["trains"][i], [(start, stop), (start + hpcbla.CHANCE_SHIFT_S, stop + hpcbla.CHANCE_SHIFT_S)], end, rng_rip)
        rows.append({"session": s["session"], "rat": s["rat"], "unit": i, "ripples": int(keep.sum()),
                     "in_reactivation_set": i in used_bla and hpcbla.usable(s),
                     "observed": real["observed"], "expected": real["null_mean"], "p_up": real["p_up"], "p_down": real["p_down"],
                     "chance_p_up": chance["p_up"], "chance_p_down": chance["p_down"]})
mod = pd.DataFrame(rows)
mod["up"] = mod.p_up <= hpcbla.RIPPLE_ALPHA
mod["down"] = mod.p_down <= hpcbla.RIPPLE_ALPHA
mod["chance_up"] = mod.chance_p_up <= hpcbla.RIPPLE_ALPHA
print(f"computed in {time.time() - started:.0f} s; BLA units", len(mod), "in", mod.session.nunique(), "sessions;",
      "units with no spike in any window", int((mod.observed == 0).sum()))
chance = mod.chance_up.mean()
n_up = int(mod.up.sum())
print("Positively modulated:", n_up, "of", len(mod), f"({mod.up.mean():.3f}); negatively: {int(mod.down.sum())}",
      f"({mod.down.mean():.3f}); chance level from the 7 s shifted windows: {chance:.3f} ({int(mod.chance_up.sum())} units)")
print("Binomial test of the positive fraction against the chance level: p =",
      f"{stats.binomtest(n_up, len(mod), max(chance, 1 / len(mod)), alternative='greater').pvalue:.2g}")
by = mod.groupby("rat").agg(units=("up", "size"), up=("up", "sum"), down=("down", "sum"), chance_up=("chance_up", "sum"))
by["up_fraction"] = by.up / by.units
by["chance_fraction"] = by.chance_up / by.units
by["p"] = [stats.binomtest(int(r.up), int(r.units), max(chance, 1 / len(mod)), alternative="greater").pvalue for r in by.itertuples()]
# Against each rat's own chance level (at least one unit, so that a chance of zero still gives a test).
by["p_own_chance"] = [stats.binomtest(int(r.up), int(r.units), max(r.chance_fraction, 1 / r.units), alternative="greater").pvalue
                      for r in by.itertuples()]
print(by.round(4).to_string())
print(mod.groupby("session").agg(units=("up", "size"), up=("up", "sum"), down=("down", "sum")).T.to_string())
print("\nThreshold variants (fraction up, chance fraction):")
for alpha in (0.05, 0.01, 0.002):
    print(f"  p <= {alpha}: {np.mean(mod.p_up <= alpha):.3f}, chance {np.mean(mod.chance_p_up <= alpha):.3f}")

In [ ]:
# Where the shifted windows land: the fraction of shifted ripple peaks that fall outside the
# Non-REM of pre- and post-sleep, for the 500 random offsets and for the 7 s shift.
rng_null = np.random.default_rng(61004)
lo, hi = (int(x / hpcbla.RIPPLE_RES_S) for x in hpcbla.RIPPLE_SHIFT_RANGE_S)
rows = []
for s in sessions:
    if s["states"] is None or s["ripples"] is None or s["units"] is None or s["session"] not in set(mod.session):
        continue
    pre, post, _ = hpcbla.sleep_epochs(s)
    nrem = hpcbla.intersect(hpcbla.intervals_of(s, "Non-REM"), np.vstack([pre, post]))
    peaks = s["ripples"]["peak"][hpcbla.in_intervals(s["ripples"]["peak"], nrem)]
    offsets = rng_null.integers(lo, hi + 1, hpcbla.RIPPLE_SHIFTS) * rng_null.choice([-1, 1], hpcbla.RIPPLE_SHIFTS) * hpcbla.RIPPLE_RES_S
    out_random = np.mean([np.mean(~hpcbla.in_intervals(peaks + o, nrem)) for o in offsets])
    rows.append({"session": s["session"], "ripples": len(peaks), "outside_nrem_random_shifts": out_random,
                 "outside_nrem_7s": np.mean(~hpcbla.in_intervals(peaks + hpcbla.CHANCE_SHIFT_S, nrem))})
null_place = pd.DataFrame(rows)
print(null_place.round(3).to_string(index=False))
print("Median over sessions: outside Non-REM with random shifts", round(float(null_place.outside_nrem_random_shifts.median()), 3),
      "; with the 7 s shift", round(float(null_place.outside_nrem_7s.median()), 3))


**Result: reproduced.** 103 of 822 BLA units (12.5%) fire more in Non-REM ripples than in the
shifted windows, against a chance level of 0.7% from the procedure run on windows shifted by 7 s
(binomial p = 9.8e-90), and the fraction is above chance in each rat (Rat08 10.3%, Rat10 4.8%,
Rat11 19.8%), against the pooled chance level and against each rat's own (Rat10 p = 0.0034 and
0.018). It is a subgroup, since the fraction is well below 1. A similar number of units (104, 12.7%)
fire less in ripples. The shifted windows
stay mostly in Non-REM (a median of 0.9% of the randomly shifted ripple peaks and 1.4% of the 7 s
shifted ones fall outside it). Units are counted once per session, so a cell recorded on several
days may be counted more than once.

## reactivation-involves-ripple-up-cells

> The hippocampus and BLA reactivations involve the subgroup of BLA cells that are positively
> modulated during sharp wave-ripples.

For each significant pattern of the run, the mean squared BLA weight of the positively modulated
units minus that of the other BLA units used.

In [ ]:
up_units = {(r.session, r.unit) for r in mod[mod.up].itertuples()}
rows = []
for r in sig.itertuples():
    is_up = np.array([(r.session, i) in up_units for i in r.bla_units])
    if is_up.any() and (~is_up).any():
        w2 = np.asarray(r.bla_weights) ** 2
        rows.append({"session": r.session, "rat": r.rat, "pattern": r.pattern, "change": r.change,
                     "n_up": int(is_up.sum()), "n_other": int((~is_up).sum()),
                     "w2_up": w2[is_up].mean(), "w2_other": w2[~is_up].mean()})
inv = pd.DataFrame(rows)
inv["diff"] = inv.w2_up - inv.w2_other
print("Patterns with both kinds of BLA unit:", len(inv), "of", len(sig), "in", inv.session.nunique(), "sessions")
print("Mean squared weight, up units: median", round(inv.w2_up.median(), 4), "; other units:", round(inv.w2_other.median(), 4),
      "; equal weights would give", round(float((1 / (inv.n_up + inv.n_other)).median()), 4))
print("Difference: median", round(inv["diff"].median(), 4), "; positive", int((inv["diff"] > 0).sum()), "of", len(inv),
      "; Wilcoxon p =", f"{wilcoxon(inv['diff']):.2g}")
print(inv.groupby("rat")["diff"].agg(["size", "median", lambda x: int((x > 0).sum())]).round(4).to_string())
reac = inv[inv.change > 0]
print("Variant, patterns with post minus pre above zero:", len(reac), "; median difference", round(reac["diff"].median(), 4),
      "; positive", int((reac["diff"] > 0).sum()), "; Wilcoxon p =", f"{wilcoxon(reac['diff']):.2g}")
print("Fraction of the BLA weight (sum of squares) on up units: median", round(float((inv.w2_up * inv.n_up / (inv.w2_up * inv.n_up + inv.w2_other * inv.n_other)).median()), 3),
      "; fraction of BLA units that are up: median", round(float((inv.n_up / (inv.n_up + inv.n_other)).median()), 3))

In [ ]:
# Firing rate. Run-epoch rate of each BLA unit used; within each pattern, each up unit is paired
# with the other BLA unit of the nearest log rate (with replacement), and the mean squared weight
# of the up units is compared with that of their matched units.
run_rate = {}
for s in used:
    _, _, run = hpcbla.sleep_epochs(s)
    for i in hpcbla.unit_sets(s)[1]:
        t = s["units"]["trains"][i]
        run_rate[(s["session"], i)] = np.count_nonzero((t >= run[0]) & (t < run[1])) / (run[1] - run[0])
unit_rates = pd.DataFrame([{"session": k[0], "unit": k[1], "rate": v, "up": k in up_units} for k, v in run_rate.items()])
print("Run-epoch rate of BLA units used (Hz): up units median", round(unit_rates[unit_rates.up].rate.median(), 2),
      f"(n = {int(unit_rates.up.sum())}); other units median", round(unit_rates[~unit_rates.up].rate.median(), 2),
      f"(n = {int((~unit_rates.up).sum())}); Mann-Whitney p = {stats.mannwhitneyu(unit_rates[unit_rates.up].rate, unit_rates[~unit_rates.up].rate).pvalue:.2g}")
rows = []
for r in sig.itertuples():
    is_up = np.array([(r.session, i) in up_units for i in r.bla_units])
    if not (is_up.any() and (~is_up).any()):
        continue
    w2 = np.asarray(r.bla_weights) ** 2
    log_rate = np.log(np.array([run_rate[(r.session, i)] for i in r.bla_units]))
    others = np.flatnonzero(~is_up)
    match = np.array([others[np.argmin(np.abs(log_rate[others] - log_rate[j]))] for j in np.flatnonzero(is_up)])
    rows.append({"session": r.session, "rat": r.rat, "pattern": r.pattern,
                 "rho_rate_w2": stats.spearmanr(log_rate, w2)[0],
                 "matched_diff": w2[is_up].mean() - w2[match].mean(),
                 "matched_log_rate_gap": float(np.median(np.abs(log_rate[is_up] - log_rate[match])))})
rate_ctl = pd.DataFrame(rows)
print("Spearman correlation of run rate and squared weight within a pattern: median", round(rate_ctl.rho_rate_w2.median(), 3),
      "; positive in", int((rate_ctl.rho_rate_w2 > 0).sum()), "of", len(rate_ctl))
print("Rate-matched difference (up minus matched other): median", round(rate_ctl.matched_diff.median(), 4), "; positive",
      int((rate_ctl.matched_diff > 0).sum()), "of", len(rate_ctl), f"; Wilcoxon p = {wilcoxon(rate_ctl.matched_diff):.2g}",
      "; median ratio of matched rates", round(float(np.exp(rate_ctl.matched_log_rate_gap.median())), 2))
inv_session = inv.groupby(["rat", "session"])["diff"].mean()
print("\nSession level: mean difference per session, median", round(inv_session.median(), 4), "; positive",
      int((inv_session > 0).sum()), "of", len(inv_session), f"; Wilcoxon p = {wilcoxon(inv_session):.2g}")
matched_session = rate_ctl.groupby("session").matched_diff.mean()
print("Session level, rate-matched: median", round(matched_session.median(), 4), "; positive", int((matched_session > 0).sum()),
      "of", len(matched_session), f"; Wilcoxon p = {wilcoxon(matched_session):.2g}")


**Result: reproduced.** In 31 significant patterns that have both kinds of BLA unit, the mean
squared weight of the ripple-up units is above that of the other BLA units (medians 0.0505 and
0.0181; difference positive in 24 of 31, p = 0.0013). The ripple-up units are a median 20% of the
BLA units of a pattern and carry 34% of its BLA weight. It holds in Rat08 (17 of 21) and Rat11 (6 of
7) and not in Rat10 (1 of 3), and in the patterns that reactivate (p = 0.0018). Firing rate
accounts for part of it: up units fire faster in the run (median 1.66 against 0.41 Hz), and
squared weight rises with rate within a pattern (median Spearman 0.46). Against other units of
matched rate the up units still carry more weight (24 of 31 patterns, p = 0.012). At session
level the difference holds (10 of 13 sessions, p = 0.027), but the rate-matched difference does
not (9 of 13, p = 0.22).

## aversive-direction-stronger

> Reactivation is stronger for the hippocampus and BLA correlation patterns of the run direction
> that held the air puff than for those of the safe direction.

Templates are built from the run bins inside laps of each direction (laps from the reward times;
the air-puff direction holds most puffs).

In [ ]:
rng_dir = np.random.default_rng(61005)
started = time.time()
directional = {s["session"]: hpcbla.reactivation(s, rng_dir, templates=("puff", "safe"), keep_traces=False) for s in used}
equal = {s["session"]: hpcbla.reactivation(s, rng_dir, templates=("puff", "safe"), equal_bins=True, keep_traces=False) for s in used}
print(f"computed in {time.time() - started:.0f} s")

def direction_table(results):
    df = pd.DataFrame([r for out in results.values() for r in out["rows"]])
    df["change"] = df.post - df.pre
    return df

dpat = direction_table(directional)
print("Template bins per session (puff, safe):", {k: (v["template_bins"].get("puff"), v["template_bins"].get("safe")) for k, v in directional.items()})

def compare(df, value="change", subset="significant"):
    d = df[df.significant] if subset == "significant" else df[df.pattern == 0]
    per = d.groupby(["rat", "session", "template"])[value].mean().unstack("template")
    per = per.dropna(subset=["puff", "safe"]) if {"puff", "safe"} <= set(per.columns) else per.iloc[0:0]
    per["diff"] = per.puff - per.safe
    return d, per

d, per = compare(dpat)
print("\nSignificant patterns: puff", int((d.template == "puff").sum()), "; safe", int((d.template == "safe").sum()))
print(per.round(4).to_string())
print("\nSessions with both:", len(per), "; median puff minus safe", round(per["diff"].median(), 4), "; positive",
      int((per["diff"] > 0).sum()), "of", len(per), "; Wilcoxon p =", f"{wilcoxon(per['diff']):.2g}")
print(per.groupby("rat")["diff"].agg(["size", "median", lambda x: int((x > 0).sum())]).round(4).to_string())

In [ ]:
rows = []
pooled = d.groupby("template").change
mw = stats.mannwhitneyu(d[d.template == "puff"].change, d[d.template == "safe"].change)
rows.append({"variant": "pooled per pattern (Mann-Whitney)", "n": f"{int((d.template == 'puff').sum())}/{int((d.template == 'safe').sum())}",
             "median_puff": round(pooled.median()["puff"], 4), "median_safe": round(pooled.median()["safe"], 4), "p": mw.pvalue})
for name, df, subset, value in (("first pattern of each direction", dpat, "first", "change"),
                                ("equal numbers of bins", direction_table(equal), "significant", "change"),
                                ("post-sleep strength alone", dpat, "significant", "post")):
    _, p = compare(df, value, subset)
    rows.append({"variant": name, "n": len(p), "median_puff": round(p.puff.median(), 4), "median_safe": round(p.safe.median(), 4),
                 "median_diff": round(p["diff"].median(), 4), "positive": int((p["diff"] > 0).sum()), "p": wilcoxon(p["diff"])})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Laps of the run epoch. A lap runs from one reward to the next at the other end, so it holds the
# time spent at the starting end as well as the run. The shortest laps of a session (its 10th
# percentile of lap duration) are taken as the time needed to travel the track; the variant below
# builds the templates from only that last part of each lap, before the reward that ends it.
rows, lap_tail = [], {}
for s in used:
    direction, counts = hpcbla.puff_direction(s)
    laps_ = hpcbla.run_laps(s)
    lap_tail[s["session"]] = float(np.percentile([b - a for a, b, _ in laps_], 10))
    row = {"session": s["session"], "puffs": int(sum(counts.values())), "puffs_in_puff_laps": counts[direction], "travel_s": round(lap_tail[s["session"]], 1)}
    for name in ("puff", "safe"):
        dur = np.array([b - a for a, b, dd in laps_ if (dd == direction) == (name == "puff")])
        row[f"{name}_laps"] = len(dur)
        row[f"{name}_median_s"] = round(float(np.median(dur)), 1)
        row[f"{name}_max_s"] = round(float(dur.max()), 1)
        row[f"{name}_bins"] = directional[s["session"]]["template_bins"][name]
        row[f"{name}_travel_fraction"] = round(float(np.minimum(dur, lap_tail[s["session"]]).sum() / dur.sum()), 3)
    rows.append(row)
lap_table = pd.DataFrame(rows)
print(lap_table.to_string(index=False))
print("Puffs inside laps of the air-puff direction, in these sessions:", int(lap_table.puffs_in_puff_laps.sum()), "of", int(lap_table.puffs.sum()))
print("Fraction of template time inside the travel part of laps: median puff", lap_table.puff_travel_fraction.median(),
      "; safe", lap_table.safe_travel_fraction.median())

rng_tail = np.random.default_rng(61006)
tail = {s["session"]: hpcbla.reactivation(s, rng_tail, templates=("puff", "safe"), lap_tail_s=lap_tail[s["session"]],
                                          keep_traces=False) for s in used}
rows = []
few = set(lap_table.session[lap_table.puffs < 5])
for name, df, keep in (("main", dpat, None), ("sessions with at least 5 puffs", dpat, few),
                       ("travel part of each lap only", direction_table(tail), None)):
    _, p = compare(df)
    if keep:
        p = p[~p.index.get_level_values("session").isin(keep)]
    rows.append({"variant": name, "sessions": len(p), "median_diff": round(p["diff"].median(), 4),
                 "positive": int((p["diff"] > 0).sum()), "p": wilcoxon(p["diff"])})
print("\nSessions left out for fewer than 5 puffs:", sorted(few))
print(pd.DataFrame(rows).to_string(index=False))


**Result: not reproduced.** Across the 15 sessions the post minus pre reactivation of the
air-puff templates exceeds that of the safe templates by a median of 0.0133, in 9 of 15 sessions,
with Wilcoxon p = 0.11, above the 0.05 of the criterion and the 0.1 of its partial outcome. By rat
the difference is positive in 6 of 9 Rat08 sessions, 2 of 3 Rat10 and 1 of 3 Rat11. The variants
lean the same way with mixed strength: the first pattern of each direction gives p = 0.048 (10
of 15), the pooled per-pattern test p = 0.083, equal numbers of bins p = 0.33 (8 of 14), post-sleep
strength alone p = 0.45, leaving out the two sessions with fewer than 5 puffs p = 0.15 (8 of 13),
and templates from the travel part of each lap only p = 0.19 (9 of 14). The laps from reward
times include the time spent at the end where a lap starts: a median of 33% of the air-puff and
43% of the safe template time lies in the travel part, and in the early Rat08 sessions laps last
up to 2310 s, so the directional templates are not templates of running alone.

## rats-learn-puff-location (blocked)

> Rats learned the location of an aversive air puff on a linear track.

Blocked as too large: the dandiset holds no position, and tracking the rat in the raw video
frames does not fit the time limit (see the data check). No criterion was written for this claim.
The lap durations below, from the reward times, are shown as a description only: they say how
long the rat took from one end to the other, not where on the track it slowed or stopped, and do
not decide an outcome.

In [ ]:
# Laps (reward to reward at the other end) in the pre-run, run and post-run epochs, by direction,
# in every file with rewards and a puff direction. A lap belongs to the epoch holding its midpoint.
rows = []
for s in sessions:
    if not all(k in s["events"] for k in ("lrw", "rrw", "puf")):
        continue
    direction, _ = hpcbla.puff_direction(s)
    if direction is None:
        continue
    for a, b, dd in hpcbla.laps(s):
        tag = [t for t, x, y in s["epochs"] if x <= (a + b) / 2 < y and t in ("prerun", "run", "postrun")]
        rows.append({"session": s["session"], "rat": s["rat"], "epoch": tag[0] if tag else "other",
                     "direction": "puff" if dd == direction else "safe", "duration": b - a})
all_laps = pd.DataFrame(rows)
print("Laps by epoch:", all_laps.epoch.value_counts().to_dict(), "(laps outside these epochs are left out)")
all_laps = all_laps[all_laps.epoch != "other"]
table = all_laps.groupby(["rat", "session", "epoch", "direction"]).duration.agg(["size", "median"]).unstack(["epoch", "direction"])
print(table.round(1).to_string())
ratio = all_laps.groupby(["session", "epoch", "direction"]).duration.median().unstack("direction")
ratio = (ratio.puff / ratio.safe).unstack("epoch")[["prerun", "run", "postrun"]]
print("\nMedian lap duration, puff direction over safe direction (reference 1):")
for epoch in ratio:
    x = ratio[epoch].dropna()
    print(f"  {epoch}: median ratio {x.median():.2f} over {len(x)} sessions, above 1 in {int((x > 1).sum())},"
          f" Wilcoxon of log ratio p = {wilcoxon(np.log(x)):.2g}")

# Air puffs by epoch, in every file with puff events.
puff_epochs = pd.DataFrame([{"session": s["session"], "epoch": next((e for e, x, y in s["epochs"] if x <= t < y), "none")}
                            for s in sessions if "puf" in s["events"] for t in s["events"]["puf"][1]])
print("\nAir puffs by epoch, over", puff_epochs.session.nunique(), "files:", puff_epochs.epoch.value_counts().to_dict())
print(f"Notebook run time: {(time.time() - notebook_started) / 60:.1f} min")


In the run, laps toward the air-puff direction take longer than laps the other way (median
ratio 1.18 over 38 sessions, above 1 in 27, p = 0.0063); in the pre-run and the post-run, when
no puffs are given (1195 of the 1197 puffs in 38 files fall in the run, 2 in the pre-sleep), the two directions take about as long (0.95 and 1.00). Slower laps where puffs
are given are what both learning and the puffs themselves would produce, so this does not show
that the rats learned where the puff was.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| `hpc-bla-reactivation-post-sleep` | reproduced |
| `reactivation-peaks-at-ripples` | reproduced |
| `bla-ripple-up-subgroup` | reproduced |
| `reactivation-involves-ripple-up-cells` | reproduced |
| `aversive-direction-stronger` | not reproduced (p = 0.11, in the claimed direction) |
| `rats-learn-puff-location` | blocked: too large (position only in the raw video frames) |

How the analysis differs from the paper's. The full text was not available, so the methods
are common choices fixed before the analysis: hippocampus and BLA as the `hpc`/`HIP` and
`bla`/`BLA` labels, 50 ms bins, joint patterns from the singular vectors of the run's
hippocampus by BLA cross-correlation matrix with a shuffle threshold, reactivation strength as the
product of the two projections, ripple windows from the `rip` annotations, laps and the air-puff
direction from the reward and puff times (there is no position), so a lap also holds the time
spent at the end where it starts. The paper's session selection,
cell classification, bin size and statistics are unknown, and 15 sessions from three rats are
used; Rat09 has no hippocampal units or ripples here.

What was found about the dandiset.
- The epochs table is in milliseconds although it says seconds; video timestamps are also in
  milliseconds, stated as seconds, and are not on the session clock.
- In ten files the units index is wrong at the first unit: unit 1 begins with the end of unit 0
  (a fall back in time after a multiple of 65536 spikes).
- Seven files have no units table, sixteen no sleep states and fourteen no ripple events; the
  Rat09 files have no hippocampal units.
- The unit region labels are undescribed short codes, in two cases for the same region in some
  files, and one file has no labels; the electrode table gives every channel as `unknown`.
- There is no position tracking, only raw video frames, so the behaviour on the track cannot be
  measured here; the `trials` table is undescribed.